# **Lab 02 - Introduction to Q-learning**

##### Copyright by UIT-NC@NT549

## **Some instructions before getting started**:
<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">

Start the Kernel: At the top right, choose <strong>Select Kernel ➞ Python Environments...</strong>

You can run all code blocks to check: From the menu bar, choose <strong>Run All</strong>.

Complete all code blocks marked with the comment <span style="font-family: monospace; font-weight: bold; color:white; background-color: green;"> ### YOU NEED TO WRITE YOUR CODE BELOW ### </span>
</div>

## Part 2: Q-learning with Custom Environment "VacuumCleaner"

In [299]:
# import Gymnasium library and alias as gym
import gymnasium as gym
import random

In [ ]:
# Build a simple custom Gymnasium environment named "VacuumCleaner-v0".
# The environment simulates a vacuum robot operating in an m x n room. The robot can
# move up, down, left, and right and automatically vacuums the cell it occupies.
# The objective is to clean all dust particles in the room. There is a single obstacle
# located at a specified cell (i, j) that the robot must avoid. Entering the obstacle
# cell yields a large negative reward and terminates the episode.
# The robot receives a positive reward when it vacuums a dirty cell. If the robot
# attempts to vacuum an already clean cell, that action receives a reduced reward
# (e.g., penalized or halved). When all dust has been cleaned, the agent receives
# a large positive bonus reward and the episode terminates.
# Action space: Discrete(4) -> {0: up, 1: down, 2: left, 3: right}
# Observation space: Dict with 'position' (x, y) and 'dust' grid (m x n binary)
import gymnasium as gym
import numpy as np
import os
import time
from IPython.display import clear_output

class VacuumCleanerEnv(gym.Env):
    def __init__(self, m=5, n=5, obstacle=(2, 2), obs_mode="encoded"):
        super(VacuumCleanerEnv, self).__init__()
        self.m = m
        self.n = n
        self.obstacle = tuple(obstacle)
        self.obs_mode = obs_mode  # "encoded" hoặc "dict"

        self.action_space = gym.spaces.Discrete(4)

        if obs_mode == "encoded":
            self.observation_space = gym.spaces.Discrete(self.m * self.n * 2)
        
        else:
            self.observation_space = gym.spaces.Dict({
                "position": 
                gym.spaces.Box(
                    low=0, 
                    high=max(self.m, self.n), 
                    shape=(2,), 
                    dtype=np.int32),
                
                "dust": 
                gym.spaces.Box(
                    low=0, 
                    high=1, 
                    shape=(self.m, self.n), 
                    dtype=np.int32)
            })

        self.reset()
        
    def _get_obs(self):
        if self.obs_mode == "encoded":
            return self.encode_state()
        else:
            return {
                "position": self.position.copy(),
                "dust": self.dust_grid.copy()
            }
        
    def encode_state(self):
        pos = self.position[0] * self.n + self.position[1]
        x, y = self.position
        nearby_dust = 0
        
        for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
            nx, ny = x+dx, y+dy
            if 0 <= nx < self.m and 0 <= ny < self.n:
                if self.dust_grid[nx, ny] == 1:
                    nearby_dust = 1
                    break

        return pos * 2 + nearby_dust

    def reset(self, *, seed=None, options=None):
        self.position = np.array([0,0], dtype=np.int32)
        
        self.dust_grid = np.ones((self.m, self.n))
        
        self.step_count = 0
        self.max_steps = 100

        self.dust_grid[self.obstacle] = 0  # obstacle cell has no dust
        self.total_reward = 0.0
        self.truncated = False
        self.terminated = False

        
        x, y = self.position
        self.dust_grid[x, y] = 0


        return self._get_obs(), {}

    def step(self, action):
        # compute candidate new position
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.step_count += 1

        if action == 0:   # Up
            candidate = self.position + np.array([-1,0])
        elif action == 1: # Down
            candidate = self.position + np.array([1,0])
        elif action == 2: # Left
            candidate = self.position + np.array([0,-1])
        elif action == 3: # Right
            candidate = self.position + np.array([0,1])
        else:
            candidate = self.position.copy()

        # boundary check
        if (0 <= candidate[0] < self.m) and (0 <= candidate[1] < self.n):
            # obstacle check
            if tuple(candidate) == self.obstacle:
                self.position = candidate.copy()
                reward = -10.0
                self.terminated = True
                self.total_reward += reward

                state = self.encode_state()
                return state, reward, True, False, {}
            else:
                self.position = candidate.copy()
        # else: stay in place

        # If the robot is on a dirty cell, give a positive reward (1.0) and mark it clean.
        # If the cell is already clean, apply a small penalty (-0.5) to discourage redundant cleaning.
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        # HERE
        x, y = self.position

        if self.dust_grid[x, y] == 1:
            reward = 3.0
            self.dust_grid[x, y] = 0
        else:
            reward = -0.5
        
        reward -= 0.05
        self.total_reward += reward

        # check if all cleaned
        if np.sum(self.dust_grid) == 0:
            reward += 20.0
            self.terminated = True


        if self.step_count >= self.max_steps:
            self.truncated = True

            
        return self._get_obs(), reward, bool(self.terminated), bool(self.truncated), {}

    def render(self, mode='human'):
        # In Jupyter notebooks, use IPython.display.clear_output to clear the cell output.
        try:
            clear_output(wait=True)
        except Exception:
            # Fallback for terminal execution
            os.system('cls' if os.name == 'nt' else 'clear')

        # Build display grid with symbols:
        # '#' obstacle, '.' dirty, ' ' clean, 'R' robot, 'X' robot on obstacle
        display = np.full((self.m, self.n), ' ', dtype='<U1')
        for i in range(self.m):
            for j in range(self.n):
                if (i, j) == self.obstacle:
                    display[i, j] = '#'
                elif self.dust_grid[i, j] == 1:
                    display[i, j] = '.'
                else:
                    display[i, j] = ' '

        x, y = int(self.position[0]), int(self.position[1])
        if (x, y) == self.obstacle:
            display[x, y] = 'X'
        else:
            display[x, y] = 'R'

        for row in display:
            print(''.join(row))
        print(f"Total reward: {self.total_reward}")
        time.sleep(0.15)

In [301]:
def round_robin_policy(env):
    x, y = 0, 0
    direction = 3  # bắt đầu đi sang phải (3 = right)

    def select_action(obs):
        nonlocal x, y, direction

        m, n = env.m, env.n  # kích thước grid

        # Nếu đang đi sang phải
        if direction == 3:
            if y < n - 1:
                y += 1
                return 3  # right
            else:
                if x < m - 1:
                    x += 1
                    direction = 2  # đổi sang trái
                    return 1  # down

        # Nếu đang đi sang trái
        elif direction == 2:
            if y > 0:
                y -= 1
                return 2  # left
            else:
                if x < m - 1:
                    x += 1
                    direction = 3  # đổi sang phải
                    return 1  # down

        return 0  # fallback (up - ít dùng)

    return select_action

In [302]:
def priority_policy():
    def select_action(obs):
        x, y = obs["position"]
        dust = obs["dust"]

        directions = [
            (-1, 0, 0),
            (1, 0, 1),
            (0, -1, 2),
            (0, 1, 3)
        ]

        for dx, dy, action in directions:
            nx, ny = x + dx, y + dy
            if 0 <= nx < dust.shape[0] and 0 <= ny < dust.shape[1]:
                if dust[nx, ny] == 1:
                    return action

        return np.random.randint(4)

    return select_action

In [303]:
def robot_policy_qlearning(state, Q, epsilon=0.1):
    if np.random.rand() < epsilon:
        return np.random.randint(Q.shape[1])   # explore
    return np.argmax(Q[state])                 # exploit


def train_q_learning(env, episodes=500, alpha=0.1, gamma=0.9, epsilon=0.1):
    Q = np.zeros((env.observation_space.n, env.action_space.n))

    rewards_per_episode = []

    for episode in range(episodes):
        state, _ = env.reset()
        terminated = False
        truncated = False
        
        total_reward = 0

        while not terminated and not truncated:
            action = robot_policy_qlearning(state, Q, epsilon)

            next_state, reward, terminated, truncated, _ = env.step(action)

            # Q update
            Q[state][action] += alpha * (
                reward + gamma * np.max(Q[next_state]) - Q[state][action]
            )

            state = next_state
            total_reward += reward
        rewards_per_episode.append(total_reward)
        
        epsilon = max(0.05, epsilon * 0.995)
        
    return Q, rewards_per_episode

def robot_policy_trained(state, Q):
    max_q = np.max(Q[state])
    actions = np.where(Q[state] == max_q)[0]
    return np.random.choice(actions)
    # return np.argmax(Q[state])

def test_agent(env, Q):
    state, _ = env.reset()
    terminated = False
    truncated = False

    env.render()

    while not terminated and not truncated:
        action = robot_policy_trained(state, Q)

        state, reward, terminated, truncated, _ = env.step(action)
        env.render()

    print("Final reward:", env.total_reward + 20.0)  # Add back the bonus for cleaning all dust

In [304]:
def robot_policy(option):
    if option == "round_robin":
        return round_robin_policy()
    elif option == "priority_based":
        return priority_policy()

In [305]:
import matplotlib.pyplot as plt

def plot_rewards(rewards):
    plt.figure()
    plt.plot(rewards)
    plt.xlabel("Episode")
    plt.ylabel("Total Reward")
    plt.title("Reward per Episode")
    plt.show()

In [ ]:
if __name__ == "__main__":

    # print("\n===== ROUND ROBIN =====")
    # env = VacuumCleanerEnv(m=5, n=5, obstacle=(2, 2), obs_mode="dict")
    # policy = robot_policy("round_robin")

    # obs, _ = env.reset()
    # terminated = truncated = False

    # while not terminated and not truncated:
    #     action = policy(obs)
    #     obs, reward, terminated, truncated, _ = env.step(action)
    #     env.render()

    # print("Total reward:", env.total_reward)

    # print("\n===== PRIORITY =====")
    # env = VacuumCleanerEnv(m=5, n=5, obstacle=(2, 2), obs_mode="dict")
    # policy = robot_policy("priority_based")

    # obs, _ = env.reset()
    # terminated = truncated = False

    # while not terminated and not truncated:
    #     action = policy(obs)
    #     obs, reward, terminated, truncated, _ = env.step(action)
    #     env.render()

    # print("Total reward:", env.total_reward)

    print("\n===== Q-LEARNING =====")
    env = VacuumCleanerEnv(m=5, n=5, obstacle=(2, 2), obs_mode="encoded")

    Q, rewards = train_q_learning(env, episodes=500)
    test_agent(env, Q)
    plot_rewards(rewards)

In [ ]:
# def save(...):
#     pass

Evaluation and Analysis